# Session 2 — Energy systems: two-node zone (M02) and storage tank (M03)

Guided notebook for Session 2. Canonical lesson: `course/session_02_energy_systems/lesson.md`.

- **M02 ZoneRC**: the zone of Session 1 split into zone air and building envelope (2R2C network). Parameters are the estimates of Bacher & Madsen (2011) for a 120 m² house.
- **M03 Storage**: a hot-water tank as $N$ fully mixed nodes with a through-flow (open system). Tank size and losses are **illustrative**; water properties are those of `Buildings.Media.Water`.

SI units inside, °C only in plots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import Stefan_Boltzmann
from scipy.integrate import solve_ivp

from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import heat_exchanger as hx
from energy_training.simulation import storage as st
from energy_training.simulation import thermal_zone as tz
from energy_training.simulation import zone_rc as zr
from energy_training.utilities import celsius_to_kelvin

apply_style()
H = 3600.0
T_amb = float(celsius_to_kelvin(0.0))

## 1. Open system: energy carried by a mass flow

For an incompressible liquid with constant $c_p$, the net energy a flow brings into a control volume is $\dot m\,c_p\,(T_{in} - T_{out})$. **Predict:** how much heat does 0.1 kg/s of water deliver when it enters at 60 °C and leaves at 30 °C?

In [ ]:
m_flow, T_in, T_out = 0.1, celsius_to_kelvin(60.0), celsius_to_kelvin(30.0)
print(f"{m_flow * st.CP_WATER * (T_in - T_out) / 1000:.2f} kW  (cp = {st.CP_WATER} J/(kg K))")

## 2. Heat transfer as resistances

Conduction, convection and (linearised) radiation each become a conductance $1/R$ between two temperatures. Radiation between a surface and large surroundings: $\dot Q = \varepsilon\sigma A (T_s^4 - T_{sur}^4) \approx h_r A (T_s - T_{sur})$ with $h_r = 4\varepsilon\sigma T_m^3$. **Predict** the order of magnitude of $h_r$ at room temperature.

In [ ]:
eps, T_m = 0.9, celsius_to_kelvin(20.0)
h_r = 4 * eps * Stefan_Boltzmann * T_m**3
print(f"h_r = {h_r:.2f} W/(m2 K)")
for dT in (1.0, 10.0, 30.0):  # error of the linearisation for a surface warmer than the room
    Ts, Tsur = T_m + dT / 2, T_m - dT / 2
    exact = eps * Stefan_Boltzmann * (Ts**4 - Tsur**4)
    print(f"dT = {dT:4.0f} K: exact {exact:7.2f} W/m2, linear {h_r * dT:7.2f} W/m2")

## 3. M02: two nodes, two time constants

$$C_z \dot T_z = \frac{T_e - T_z}{R_{ze}} + \dot Q, \qquad C_e \dot T_e = \frac{T_z - T_e}{R_{ze}} + \frac{T_{amb} - T_e}{R_{ea}}$$

**Predict:** which time constants do you expect, given $R_{ze}C_z$ and $R_{ea}C_e$?

In [ ]:
p2 = zr.ZoneRCParams()
print(f"C_z = {p2.C_z:.3g} J/K, C_e = {p2.C_e:.3g} J/K")
print(f"R_ze = {p2.R_ze:.3g} K/W, R_ea = {p2.R_ea:.3g} K/W")
print(f"R_ze C_z = {p2.R_ze * p2.C_z / H:.2f} h,  R_ea C_e = {p2.R_ea * p2.C_e / H:.1f} h")
A, B, Cy, D = zr.state_space(p2)
print("A =", A)
print("time constants from the eigenvalues [h]:", np.round(zr.time_constants(p2) / H, 3))

The eigenvalues are not simply $-1/(R C)$ of each node: the nodes are coupled. Now a heater step from 1 kW to 4 kW, starting in equilibrium, compared with M01 using the same total capacity and conductance.

In [ ]:
Q0, Q1 = 1000.0, 4000.0
x0 = zr.steady_state(Q0, T_amb, p2)
t = np.linspace(0, 96 * H, 1153)
res2 = zr.simulate((0, 96 * H), x0, Q1, T_amb, p2, t_eval=t)
C_tot, UA_tot = zr.lumped_equivalent(p2)
m01 = tz.analytic_step(t, x0[0], Q1, T_amb, tz.ThermalZoneParams(C=C_tot, UA=UA_tot))

fig, ax = plt.subplots()
ax.plot(hours(t), degc(res2.T_z), label="M02 zone")
ax.plot(hours(t), degc(res2.T_e), label="M02 envelope")
ax.plot(hours(t), degc(m01), ls="--", label="M01, same C and UA")
ax.set(xlabel="time [h]", ylabel="temperature [°C]")
ax.legend();

In [ ]:
eb = zr.energy_balance(res2, p2)
print({k: f"{v / 3.6e6:.4f} kWh" for k, v in eb.items()})

## 4. M03: storage tank with $N$ mixed nodes

Node $k$ (node 1 at the top), charging from the top:

$$m_k c_p \dot T_k = \dot m\,c_p\,(T_{k-1} - T_k) - \frac{UA_{loss}}{N}(T_k - T_{env}),\qquad T_0 \equiv T_{in}$$

**Predict:** how long does the water stay in a 300 L tank at 0.1 kg/s?

In [ ]:
p3 = st.StorageParams()  # 300 L, N = 10, UA_loss = 2 W/K (illustrative)
m = 0.1
T_cold, T_hot, T_env = (float(celsius_to_kelvin(x)) for x in (30.0, 60.0, 20.0))
tau_res = st.residence_time(m, p3)
print(
    f"node mass {p3.m_node:.1f} kg, mean residence time {tau_res / H:.2f} h, "
    f"loss time constant {st.loss_time_constant(p3) / H:.0f} h"
)

t = np.linspace(0, 3 * H, 541)
res3 = st.simulate((0, 3 * H), T_cold, m, T_hot, T_env, p3, t_eval=t)
fig, ax = plt.subplots()
for k in (0, 4, 9):
    ax.plot(hours(t), degc(res3.T[:, k]), label=f"node {k + 1}")
ax.set(xlabel="time [h]", ylabel="temperature [°C]")
ax.legend();

In [ ]:
eb3 = st.energy_balance(res3, p3)
throughput = sum(abs(v) for k, v in eb3.items() if k != "residual")
print({k: f"{v / 3.6e6:.4f} kWh" for k, v in eb3.items()})
print(f"relative residual {abs(eb3['residual']) / throughput:.1e}")

### How many nodes?

With losses neglected, the outlet after an inlet step has an exact solution (mixed tanks in series). **Predict** how the outlet changes with $N$.

In [ ]:
s = np.linspace(0, 2.5, 501)  # time / residence time
fig, ax = plt.subplots()
for N in (1, 3, 10, 50):
    pN = st.StorageParams(N=N, UA_loss=0.0)
    ax.plot(
        s,
        degc(st.analytic_series_step(s * st.residence_time(m, pN), N, T_cold, T_hot, m, pN)),
        label=f"N = {N}",
    )
ax.axvline(1.0, color="k", lw=1, ls=":")
ax.set(xlabel="time / mean residence time", ylabel="outlet temperature [°C]")
ax.legend();

More nodes give a sharper front, but each node gets faster: the node time constant is the residence time divided by $N$, while the losses act over days. That ratio makes the equations **stiff** (Session 4). Count the right-hand-side calls of an explicit and an implicit solver over one day (the implicit one also calls it to build its Jacobian; `sol.nfev` does not count those):

In [ ]:
def count_calls(N, method):
    """Right-hand-side calls over one day, including those used to build the Jacobian."""
    pN, calls = st.StorageParams(N=N), [0]

    def f(t, T):
        calls[0] += 1
        return st.rhs(t, T, m, T_hot, T_env, pN)

    solve_ivp(f, (0, 24 * H), np.full(N, T_cold), method=method, rtol=1e-6, atol=1e-6)
    return calls[0]


for N in (10, 50, 200):
    print(
        f"N = {N:3d}: RK45 {count_calls(N, 'RK45'):6d} calls, BDF {count_calls(N, 'BDF'):4d} calls"
    )

## 5. A steady-state component: heat exchanger (ε-NTU)

$\dot Q = \varepsilon\,C_{min}\,(T_{h,in} - T_{c,in})$, $NTU = UA/C_{min}$. A counterflow plate exchanger, $UA$ = 1500 W/K, hot side 0.1 kg/s at 60 °C, cold side 0.15 kg/s at 30 °C:

In [ ]:
C_h, C_c = 0.1 * st.CP_WATER, 0.15 * st.CP_WATER
for arr in ("counterflow", "parallel"):
    Qhx = hx.heat_flow(1500.0, C_h, C_c, T_hot, T_cold, arr)
    print(f"{arr:12s}: Q = {Qhx / 1000:.2f} kW, hot outlet {degc(T_hot - Qhx / C_h):.1f} degC")

This model has no state: it answers instantly. Using it inside a dynamic model assumes the exchanger's own heat capacity is negligible compared with the time scales of interest.

## 6. Your turn

- **G2.1** (guided): `exercises/session_02/G2.1_storage_energy_balance.md`
- **I2.1** (independent): `exercises/session_02/I2.1_dominant_time_constants.md`

**Takeaways.** Each node is a control volume with its own energy balance; connections are conductances or mass flows. More states bring more time scales — read them from the eigenvalues. The number of nodes is a modelling decision with numerical consequences.